In [1]:
"""
Resample an anisotropic mask TIFF stack to isotropic 10nm voxels.

Assumptions:
 - Input TIFF path: /mnt/data/Mitolysosome_consensus_02.tif
 - Original voxel spacing: x=1 nm, y=1 nm, z=70 nm
 - Target spacing: x=10 nm, y=10 nm, z=10 nm
 - Each mask has a single intensity value (integer). Background assumed 0.
 - Output path: /mnt/data/Mitolysosome_consensus_02_resampled_10nm.tif

Pipeline:
 1) Read stack (z,y,x).
 2) Compute zoom factors: new_voxels = old_voxels * (orig_spacing / target_spacing)
    For our case: z zoom = 70/10 = 7 (upsample), y/x zoom = 1/10 = 0.1 (downsample by 10).
 3) Center-crop Y and X so they are divisible by 10 (required for block-average).
 4) For each label != 0:
    a) Make binary mask (float32).
    b) Upsample Z only with scipy.ndimage.zoom( order=3 ) -> smooth Z interpolation.
    c) Block-average in X/Y by reshaping into (y_out, ds, x_out, ds) and mean over the ds dims.
    d) Store the result as a soft map (float).
 5) Argmax over labels per voxel -> assign label; voxels with low max confidence are set to background (0).
 6) Save TIFF.

Notes:
 - If you'd rather preserve label boundaries exactly (no smoothing), use nearest-neighbor in Z and max-voting without smoothing.
 - For extremely large data, consider using dask / chunking or memmap to avoid memory spikes.
"""

import numpy as np
import tifffile
from scipy import ndimage
import os

# ---------- User-editable paths / params ----------
IN_PATH  = "Mitolysosome_consensus_02.tif"
OUT_PATH = "Mitolysosome_consensus_02_resampled_10nm.tif"

# original spacings in nm (z, y, x)
orig_spacing = np.array([70.0, 1.0, 1.0])
target_spacing = np.array([10.0, 10.0, 10.0])
# background label
BACKGROUND = 0
# confidence threshold below which we set background
BG_CONF_THRESHOLD = 0.1
# -------------------------------------------------

# Read input
print("Reading:", IN_PATH)
stack = tifffile.imread(IN_PATH)
stack = np.asarray(stack)
if stack.ndim != 3:
    raise ValueError("Expected a 3D TIFF stack with shape (z, y, x). Got shape: {}".format(stack.shape))
z_in, y_in, x_in = stack.shape
print("Input shape (z,y,x) =", stack.shape, "dtype=", stack.dtype)

# compute zoom factors (z,y,x)
zoom = (orig_spacing / target_spacing).tolist()
print("Zoom factors (z,y,x) =", zoom)
z_zoom = zoom[0]
y_zoom = zoom[1]  # expected 0.1
x_zoom = zoom[2]  # expected 0.1

# output sizes (rounded)
z_out = int(round(z_in * z_zoom))
y_out = int(round(y_in * y_zoom))
x_out = int(round(x_in * x_zoom))
print("Estimated output shape (z,y,x) =", (z_out, y_out, x_out))

# compute integer downsample factors in y,x
ds_y = int(round(1.0 / y_zoom))
ds_x = int(round(1.0 / x_zoom))
assert ds_y > 0 and ds_x > 0, "Invalid downsample factors"
print("Downsample factors (y,x) =", (ds_y, ds_x))

# center-crop Y and X so they are multiples of ds_y, ds_x (required for block average)
y_keep = y_out * ds_y
x_keep = x_out * ds_x
if y_keep > y_in or x_keep > x_in:
    # fallback: compute nearest floor multiple instead
    y_keep = (y_in // ds_y) * ds_y
    x_keep = (x_in // ds_x) * ds_x
    y_out = y_keep // ds_y
    x_out = x_keep // ds_x
    print("Adjusted crop sizes to available input. New output (y,x) =", (y_out, x_out))

y_pad = y_in - y_keep
x_pad = x_in - x_keep
y0 = y_pad // 2
x0 = x_pad // 2
y1 = y0 + y_keep
x1 = x0 + x_keep
print("Cropping Y:{}:{}  X:{}:{}".format(y0, y1, x0, x1))

stack_c = stack[:, y0:y1, x0:x1]
print("Cropped input shape:", stack_c.shape)

# labels (skip background)
labels = [int(v) for v in np.unique(stack_c) if int(v) != BACKGROUND]
if len(labels) == 0:
    raise ValueError("No non-background labels found.")
print("Found labels:", labels)

# allocate soft stack (labels, z_out, y_out, x_out)
soft = np.zeros((len(labels), z_out, y_out, x_out), dtype=np.float32)

for i, lab in enumerate(labels):
    print(f"Processing label {lab} ({i+1}/{len(labels)}) ...")
    # binary mask
    binary = (stack_c == lab).astype(np.float32)  # shape (z_in, y_keep, x_keep)
    # upsample in z only (order=3 for smooth z interpolation)
    up = ndimage.zoom(binary, zoom=(z_zoom, 1.0, 1.0), order=3, mode='nearest', prefilter=True)
    print("  after Z upsample:", up.shape)
    # sanity check sizes: up.shape should be (z_out, y_keep, x_keep)
    if up.shape[0] != z_out:
        # if rounding differences create off-by-one, adjust by cropping/pad first axis
        if up.shape[0] > z_out:
            up = up[:z_out, :, :]
        else:
            padz = z_out - up.shape[0]
            up = np.pad(up, ((0, padz), (0, 0), (0, 0)), mode='constant', constant_values=0)
    # block-average in y and x
    # reshape: (z_out, y_out, ds_y, x_out, ds_x)
    z_u, y_u, x_u = up.shape
    assert y_u == y_out * ds_y and x_u == x_out * ds_x, "Unexpected sizes for block reduction"
    up_reshaped = up.reshape(z_u, y_out, ds_y, x_out, ds_x)
    block_mean = up_reshaped.mean(axis=(2, 4))  # (z_out, y_out, x_out)
    soft[i] = block_mean.astype(np.float32)
    print("  stored soft map min/max:", soft[i].min(), soft[i].max())

# combine via argmax
print("Combining soft maps by argmax...")
label_idx = np.argmax(soft, axis=0)   # picks index of best label per voxel
max_vals = soft.max(axis=0)
out = np.zeros((z_out, y_out, x_out), dtype=stack.dtype)
for i, lab in enumerate(labels):
    out[label_idx == i] = lab
# set background where confidence low
out[max_vals < BG_CONF_THRESHOLD] = BACKGROUND

# Save output
print("Saving to:", OUT_PATH)
tifffile.imwrite(OUT_PATH, out, photometric='minisblack', bigtiff=True)
print("Done. Output shape:", out.shape)


Reading: Mitolysosome_consensus_02.tif
Input shape (z,y,x) = (22, 4092, 3222) dtype= uint32
Zoom factors (z,y,x) = [7.0, 0.1, 0.1]
Estimated output shape (z,y,x) = (154, 409, 322)
Downsample factors (y,x) = (10, 10)
Cropping Y:1:4091  X:1:3221
Cropped input shape: (22, 4090, 3220)
Found labels: [3]
Processing label 3 (1/1) ...
  after Z upsample: (154, 4090, 3220)
  stored soft map min/max: -0.26201463 1.2610452
Combining soft maps by argmax...
Saving to: Mitolysosome_consensus_02_resampled_10nm.tif
Done. Output shape: (154, 409, 322)
